# EM Volume Exploration

**Author:** Kevin Druciak (kevintdruciak@gmail.com)

Loading, inspecting, and visualizing electron microscopy (EM) volumes using standard Python libraries. This notebook demonstrates the data exploration stage of the connectomics pipeline -- understanding volume geometry, intensity distributions, and spatial structure before running segmentation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

try:
    import nrrd
    HAS_NRRD = True
except ImportError:
    HAS_NRRD = False

try:
    import nibabel as nib
    HAS_NIBABEL = True
except ImportError:
    HAS_NIBABEL = False

## Load Volume

In [ ]:
VOLUME_PATH = "../sample_data/mini_em_stack.nrrd"

def load_volume(path):
    if path.endswith(".nrrd") and HAS_NRRD:
        data, header = nrrd.read(path)
        spacing = np.abs(np.diag(header.get("space directions", np.eye(3))))
        origin = header.get("space origin", np.zeros(3))
        return data, {"spacing": spacing, "origin": origin, "raw_header": header}
    elif HAS_NIBABEL:
        img = nib.load(path)
        data = np.asarray(img.dataobj)
        spacing = img.header.get_zooms()[:3]
        origin = img.affine[:3, 3]
        return data, {"spacing": spacing, "origin": origin, "affine": img.affine}
    else:
        raise ImportError("Install pynrrd or nibabel to load volumes.")

import os
if not os.path.exists(VOLUME_PATH):
    np.random.seed(42)
    volume = np.random.randint(80, 200, size=(64, 128, 128), dtype=np.uint8)
    volume[:, ::16, :] = np.random.randint(10, 60, size=(64, 8, 128), dtype=np.uint8)
    volume[:, :, ::16] = np.random.randint(10, 60, size=(64, 128, 8), dtype=np.uint8)
    meta = {"spacing": np.array([1.0, 1.0, 1.0]), "origin": np.zeros(3)}
else:
    volume, meta = load_volume(VOLUME_PATH)

print(f"Shape: {volume.shape}")
print(f"Dtype: {volume.dtype}")
print(f"Spacing: {meta['spacing']}")
print(f"Intensity range: [{volume.min()}, {volume.max()}]")
print(f"Memory: {volume.nbytes / 1e6:.1f} MB")

## Orthogonal Slices

In [ ]:
def show_orthogonal_slices(vol, title="EM Volume"):
    z, y, x = [s // 2 for s in vol.shape]
    fig = plt.figure(figsize=(14, 4))
    gs = GridSpec(1, 3, figure=fig)

    for i, (slc, label) in enumerate([
        (vol[z, :, :], f"Axial (z={z})"),
        (vol[:, y, :], f"Coronal (y={y})"),
        (vol[:, :, x], f"Sagittal (x={x})"),
    ]):
        ax = fig.add_subplot(gs[0, i])
        ax.imshow(slc, cmap="gray", aspect="equal")
        ax.set_title(label)
        ax.axis("off")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

show_orthogonal_slices(volume)

## Intensity Histogram

The bimodal distribution is characteristic of EM tissue images: a dark peak corresponding to membranes and a bright peak corresponding to cell interiors. The threshold for membrane detection is chosen in the valley between these peaks.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(volume.ravel(), bins=128, color="steelblue", edgecolor="none", alpha=0.8)
ax.set_xlabel("Intensity")
ax.set_ylabel("Voxel Count")
ax.set_title("EM Volume Intensity Histogram")
ax.axvline(x=128, color="red", linestyle="--", label="Membrane threshold (128)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean: {volume.mean():.1f}  Std: {volume.std():.1f}  Median: {np.median(volume):.1f}")

## Slice Montage

In [ ]:
n_slices = min(12, volume.shape[0])
indices = np.linspace(0, volume.shape[0] - 1, n_slices, dtype=int)
cols = 4
rows = int(np.ceil(n_slices / cols))

fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes = axes.flatten()

for i, idx in enumerate(indices):
    axes[i].imshow(volume[idx], cmap="gray")
    axes[i].set_title(f"z={idx}", fontsize=9)
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Axial Slice Montage", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()